In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

driver = webdriver.Chrome()
driver.maximize_window()
wait = WebDriverWait(driver, 10)

driver.get("http://localhost:5173/")
driver.execute_script("window.localStorage.clear(); window.sessionStorage.clear();")
driver.get("http://localhost:5173/login")

wait.until(EC.presence_of_element_located((By.ID, "username")))
driver.find_element(By.ID, "username").send_keys("shawon@gmail.com")
driver.find_element(By.ID, "password").send_keys("12345678")
driver.find_element(By.ID, "sign-in-btn").click()

time.sleep(3)
print("URL:", driver.current_url)
print("Page text:", driver.find_element(By.TAG_NAME, "body").text[:200])

In [ ]:
try:
    # Go to POS / Sales (sidebar button, verified in AppShell.jsx)
    wait.until(EC.element_to_be_clickable((By.XPATH, "//aside//button[contains(., 'POS / Sales')]"))).click()
    wait.until(EC.visibility_of_element_located((By.XPATH, "//h2[text()='POS / Sales']")))

    # Add the first available medicine to the cart (Add button, verified in CatalogTable.jsx)
    wait.until(EC.presence_of_element_located((By.XPATH, "//tr[contains(@class, 'pos-row')]")))
    add_buttons = [b for b in driver.find_elements(By.XPATH, "//tr[contains(@class, 'pos-row')]//button[contains(., 'Add')]") if b.is_displayed() and b.is_enabled()]
    assert add_buttons, "No addable medicine found in the catalog."
    med_name = add_buttons[0].find_element(By.XPATH, "./ancestor::tr[1]").text.split("\n")[0]
    print("Medicine:", med_name)
    add_buttons[0].click()
    time.sleep(2)

    # Complete the sale
    wait.until(EC.element_to_be_clickable((By.XPATH, "//button[contains(., 'Complete Sale') and not(contains(., 'Reviewed'))]"))).click()
    time.sleep(3)
    approve = [b for b in driver.find_elements(By.XPATH, "//button[contains(., 'Reviewed')]") if b.is_displayed()]
    if approve:
        print("Approval modal appeared, confirming...")
        approve[0].click()
        time.sleep(3)

    # Verify the real receipt modal (verified in ReceiptModal.jsx)
    wait.until(EC.visibility_of_element_located((By.XPATH, "//*[text()='Sale Completed']")))
    body = driver.find_element(By.TAG_NAME, "body").text
    invoice = driver.find_element(By.XPATH, "//*[contains(text(), 'Invoice #')]").text
    assert med_name in body, "Medicine name missing from the receipt."
    assert "Customer:" in body, "Customer info missing from the receipt."
    assert "Total Paid" in body, "Total missing from the receipt."
    new_sale = [b for b in driver.find_elements(By.XPATH, "//button[contains(., 'Start New Sale')]") if b.is_displayed()]
    assert new_sale, "'Start New Sale' button missing from the receipt."
    has_print = any(b.is_displayed() for b in driver.find_elements(By.XPATH, "//button[contains(., 'Print')]"))
    print("Print Receipt button exists:", has_print, "(no physical printing done)")

    print("Receipt:", invoice)
    print("Receipt text:", body[body.find('Sale Completed'):body.find('Sale Completed') + 500])
    print("Current URL:", driver.current_url)
    print("PASS: Receipt")
except Exception as e:
    print("FAIL: Receipt")
    print("Error:", e)
    driver.save_screenshot("20_receipt_FAIL.png")

In [ ]:
driver.quit()